In [1]:
from datasets import load_dataset
import pandas as pd
from collections import defaultdict, Counter
import numpy as np
import torch
import torch.nn as nn
import json
import re
import kenlm
from tqdm import tqdm
import math
import lightgbm as lgb
import os
import warnings

C:\Users\ADMIN\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 1. Load các file cần thiết và dataset
## 1.1 Load vocab

In [2]:
vocab = []
with open(r"D:\NLP\project\vocabulary.txt", 'r', encoding='utf-8') as f:
    vocab = f.read().splitlines()

# Lọc các từ giống nhau
vocab = list(dict.fromkeys(vocab))

# Ánh xạ word và idx
word_to_idx = {word: i for i, word in enumerate(vocab)}

## 1.2 Load các stopwords của tiếng Việt

In [3]:
with open(r"D:\NLP\project\vietnamese-stopwords.txt", 'r', encoding='utf-8') as f:
    stopwords = f.read().splitlines()
stopword = set(stopwords)

## 1.3 Load file tách âm tiếng Việt
1 âm tiếng Việt -> tách ra nguyên âm - phụ âm - dấu câu theo cách gõ unicode

In [4]:
with open(r"D:\NLP\project\telex.txt", "r", encoding="utf-8") as f:
    telex = f.read()

telex = re.sub(r',\s*}', '\n}', telex)
telex = json.loads(telex)
telex

{'ă': ['a', 'w', ''],
 'â': ['a', 'a', ''],
 'ê': ['e', 'e', ''],
 'ô': ['o', 'o', ''],
 'ơ': ['o', 'w', ''],
 'ư': ['u', 'w', ''],
 'đ': ['d', 'd', ''],
 'á': ['a', '', 's'],
 'à': ['a', '', 'f'],
 'ã': ['a', '', 'x'],
 'ả': ['a', '', 'r'],
 'ạ': ['a', '', 'j'],
 'ắ': ['a', 'w', 's'],
 'ằ': ['a', 'w', 'f'],
 'ẵ': ['a', 'w', 'x'],
 'ẳ': ['a', 'w', 'r'],
 'ặ': ['a', 'w', 'j'],
 'ấ': ['a', 'a', 's'],
 'ầ': ['a', 'a', 'f'],
 'ẫ': ['a', 'a', 'x'],
 'ẩ': ['a', 'a', 'r'],
 'ậ': ['a', 'a', 'j'],
 'é': ['e', '', 's'],
 'è': ['e', '', 'f'],
 'ẽ': ['e', '', 'x'],
 'ẻ': ['e', '', 'r'],
 'ẹ': ['e', '', 'j'],
 'ế': ['e', 'e', 's'],
 'ề': ['e', 'e', 'f'],
 'ễ': ['e', 'e', 'x'],
 'ể': ['e', 'e', 'r'],
 'ệ': ['e', 'e', 'j'],
 'í': ['i', '', 's'],
 'ì': ['i', '', 'f'],
 'ĩ': ['i', '', 'x'],
 'ỉ': ['i', '', 'r'],
 'ị': ['i', '', 'j'],
 'ó': ['o', '', 's'],
 'ò': ['o', '', 'f'],
 'õ': ['o', '', 'x'],
 'ỏ': ['o', '', 'r'],
 'ọ': ['o', '', 'j'],
 'ố': ['o', 'o', 's'],
 'ồ': ['o', 'o', 'f'],
 'ỗ': ['o', 'o'

## 1.4 Load dataset

In [5]:
dataset = load_dataset("yammdd/vietnamese-error-correction-corpus")
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['input', 'target'],
        num_rows: 56445
    })
    validation: Dataset({
        features: ['input', 'target'],
        num_rows: 7056
    })
    test: Dataset({
        features: ['input', 'target'],
        num_rows: 7056
    })
})


# 2. Xử lý dataset
Sử dụng các hàm từ file process_dataset.py để xử lý dataset

In [6]:
from process_dataset import process_dataset, split_data

## 2.1 Thay đổi bộ dataset cho phù hợp với định hướng
Chuyển tất cả từ có viết hoa thành chữ thường.\
Không xử lý số nên bỏ hết số.\
Không xử lý dấu câu nên bỏ hết dấu câu.\
Không xử lý lỗi dấu cách, bỏ các dòng có độ dài 2 bên không bằng nhau.\
Không xử lý các từ không có trong vocab(tiếng Anh, tên riêng, vv), chuyển các từ đó thành từ đúng bên target.

In [7]:
df = dataset.map(process_dataset,
                batched=True, 
                remove_columns=dataset['train'].column_names,
                fn_kwargs={"word_to_idx": word_to_idx})

## 2.2 Chia dataset
Tách dataset thành 2 loại chính: chính tả + viết tắt / không dấu.\
Mục đích: dễ train, dễ kiểm soát, tập trung từng loại dễ hơn, ít hơn so với data gốc nên nhanh hơn. Sau khi xong sẽ gộp lại tính chung.

In [8]:
df1, df2 = split_data(df)

## 2.3 Chia train/test/valid
Chia tập train/test/valid.\
Note: df_valid dùng để thử các kết quả model trước khi áp dụng lần cuối với test

In [9]:
df_train = pd.DataFrame(df['train'])
df_test = pd.DataFrame(df['test'])
df_valid = pd.DataFrame(df['validation'])

df1_train = pd.DataFrame(df1['train'])
df1_test = pd.DataFrame(df1['test'])
df1_valid = pd.DataFrame(df1['validation'])

df2_train = pd.DataFrame(df2['train'])
df2_test = pd.DataFrame(df2['test'])
df2_valid = pd.DataFrame(df2['validation'])

# 3. Sửa lỗi viết tắt, teencode
Hướng đi: 
- Load 1 danh sách và mapping các từ viết tắt/teencode phổ biến
- Mỗi khi gặp 1 từ trong danh sách -> chuyển sang dạng đúng của nó như trong file
## 3.1 Load file các từ viết tắt/teencode 

In [10]:
abbreviation_dict = {}

with open(r"D:\NLP\project\teen_code.txt", 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        
        parts = line.split(maxsplit=1)
        shortcut, full_word = parts[0].lower(), parts[1].lower()
        abbreviation_dict[shortcut] = full_word

## 3.2 Tạo hàm quét và sửa từ viết tắt

In [11]:
def replace_abbreviations(sentence):
    words = sentence.lower().split()
    
    for i, word in enumerate(words):
        if word in abbreviation_dict:
            if len(abbreviation_dict[word].split()) == 1:
                words[i] = abbreviation_dict[word]
            
    return " ".join(words)

Test hàm

In [12]:
sentence = "hôm nay có đi ăn ko"
print(replace_abbreviations(sentence))

hôm nay có đi ăn không


# 4. Phát hiện từ lỗi chính tả hoặc sai ngữ cảnh
## 4.1 Model Tri-gram bằng kenlm
Ta train trên phần 'target' của tập train dataset ban đầu, train ở file train_skipgram.ipynb.

In [13]:
model_lm = kenlm.Model(r"D:/NLP/project/trigram.bin")

## 4.2 Phát hiện các vị trí lỗi:
Bằng cách tra từ không thuộc từ điển và N-gram (tri-gram) của từ quá thấp thì từ đó sẽ lỗi.\
Ta sẽ tìm các tham số để tối đa khả năng bắt lỗi của hàm thông qua file find_parameter_detect.ipynb.\
Sử dụng hàm detect_error_word từ detect_error.py

In [14]:
from detect_error import detect_error

Test hàm detect_error

In [15]:
sentence = "nhiều giãi pháp đã áp dụng thành coong trong thực tiễn có doanh thu tương dối lớn phục vụ không chỏ cho doanh nghiệp mà còn cho đời sóng xã hội"
error_indices = detect_error(sentence, model_lm)
print(error_indices)
words = sentence.split()
for i in error_indices:
    print(words[i])

[1, 5, 7]
giãi
dụng
coong


# 5. Sửa lỗi chính tả, ngữ cảnh
Hướng đi: với mỗi từ được cho là lỗi
- Tạo danh sách các ứng viên các từ đúng với từ đó 
- Trích xuất đặc trưng của từng ứng viên:
    1. Điểm tri-gram của nó với các từ xung quanh (model kenlm)
    2. Điểm similarity của nó với các từ xung quanh (model skip-gram)
    3. Tần suất của nó
    4. Tần suất của từ ghép 2 của nó và từ kế bên
    5. Tần suất của từ ghép 3 của nó và các từ kế bên
    6. Edit_distance của nó và từ sai
    7. Tỉ lệ độ dài của từ viết sai và nó
    8. Thứ tự xuất hiện của nó trong hàm lookup
- Sử dụng Lightgbm để chọn ứng viên có điểm cao nhất

## 5.1 Tạo các ứng viên cho 1 từ
### 5.1.1 Đếm tần suất
Tạo 1 Dictionary là counts để tính số lần xuất hiện của mỗi từ trong vocab (chỉ xử lý từ đơn, từ ghép 2 và từ ghép 3)

In [16]:
counts_1 = Counter() 
counts_2 = Counter() 
counts_3 = Counter() 

for sentence in df_train['target']:
    # Tách từ theo khoảng trắng
    tokens = str(sentence).lower().split()
    if not tokens:
        continue
        
    # Đếm từ đơn 
    counts_1.update(tokens)
    
    # Đếm từ ghép 2 bằng cách trượt cửa sổ cặp đôi
    # zip(tokens, tokens[1:]) tạo ra các cặp liên tiếp
    bigrams = [" ".join(p) for p in zip(tokens, tokens[1:])]
    counts_2.update(bigrams)
    
    # Đếm từ ghép 3 bằng cách trượt cửa sổ bộ ba
    trigrams = [" ".join(t) for t in zip(tokens, tokens[1:], tokens[2:])]
    counts_3.update(trigrams)

### 5.1.2 Tạo từ điển lưu toàn bộ biến thể xóa
Tạo danh sách các từ thiếu từ 0 đến k kí tự trong vocab bằng hàm get_deletes dựa trên create_telex_form của file create_candidates.py.\
Tạo 1 Dictionary (Hash Map) để lưu toàn bộ biến thể xóa.

In [17]:
from create_candidates import create_telex_form, get_deletes

sym_dict = defaultdict(list)

for word in vocab:
    length = word.split(' ')
    if len(length) > 1:
        continue
    # Lấy các biến thể và từ gốc của 1 từ
    base_forms = [word] + create_telex_form(word, telex)

    for form in base_forms:
        # Lưu form này (distance 0)
        if word not in sym_dict[form]:
            sym_dict[form].append(word)

        # Tạo deletes cho từng form và lưu vào từ điển
        variant_list = get_deletes(form)
        for variant in variant_list:
            # Map biến thể tới từ gốc 'word' hiện tại nếu chưa map
            if word not in sym_dict[variant]:
                sym_dict[variant].append(word)

### 5.1.3 Tạo ứng viên 
Sử dụng các hàm từ lookup.py.\
Sử dụng Symmetric Delete: Thuật toán tạo các biến thể có thể là từ đúng của 1 từ.\
Chỉ sử dụng phép Xóa (Delete) để thu hẹp không gian tìm kiếm.\
Nếu hai từ có khoảng cách chỉnh sửa (Edit Distance) là $k$, chúng sẽ có chung ít nhất một biến thể "xóa đi $n$ ký tự" ($n \le k$).\
Tra cứu biến thể xóa để xem có những từ nào cũng có biến thể xóa đó.

In [18]:
from create_candidates import lookup, edit_distance_telex

Test hàm lookup

In [19]:
lookup("trwuwowng", sym_dict, telex, word_to_idx, counts_1)

['trướng', 'trường', 'trưởng', 'trương', 'trượng', 'tướng']

## 5.2 Trích xuất đặc trưng
### 5.2.1 Model skip-gram
Ta train trên phần 'target' của tập train dataset ban đầu, train ở file train_skipgram.ipynb.\
Sau đó ta sử dụng model skip-gram để tạo hàm tính toán độ liên quan của 2 từ

In [20]:
class SkipGram(nn.Module):

    def __init__(self, vocab_size, embed_dim):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.linear = nn.Linear(embed_dim, vocab_size)

    def forward(self, x):

        embed = self.embedding(x)
        out = self.linear(embed)

        return out

In [21]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EMBED_DIM = 300
VOCAB_SIZE = len(vocab)

model_skipgram = SkipGram(VOCAB_SIZE, EMBED_DIM).to(device)

model_path = r"D:/NLP/project/model_skipgram.pth"
model_skipgram.load_state_dict(torch.load(model_path, weights_only=True))

<All keys matched successfully>

In [22]:
embeddings = model_skipgram.embedding.weight.data
embedding_matrix = embeddings.cpu().numpy()

### 5.2.2 Hàm trích xuất đặc trưng đơn lẻ
Trích xuất 5 đặc trưng (trừ đặc trưng edit distance) của 1 ứng viên bằng các hàm trong file extract_feature.py

In [23]:
from extract_feature import (extract_similarity_features, 
                             extract_ngram_counts_feature, 
                             extract_length_ratio_feature,
                             extract_kenlm_feature)

### 5.2.3 Trích xuất đặc trưng toàn bộ ứng viên
Tạo hàm trích xuất đặc trưng toàn bộ ứng viên, trả về list các (candidate, features)

In [24]:
norms = np.linalg.norm(embedding_matrix, axis=1, keepdims=True)
norms[norms == 0] = 1 
norm_embedding_matrix = embedding_matrix / norms

def extract_candidates_and_features(error_word, sentence_words, error_idx, error_indices, window_size=3):
    n_words = len(sentence_words)
    
    # Chuẩn bị ngữ cảnh cho đặc trưng tri-gram (kenlm)
    local_start = max(0, error_idx - window_size)
    local_end = min(n_words, error_idx + window_size + 1)
    
    prefix_words = sentence_words[local_start:error_idx]
    suffix_words = sentence_words[error_idx + 1:local_end]
    prefix_str = " ".join(prefix_words) + " " if prefix_words else ""
    suffix_str = " " + " ".join(suffix_words) if suffix_words else ""

    # Chuẩn bị ngữ cảnh cho đặc trưng tần suất từ ghép 2 và 3
    prev_word = sentence_words[error_idx - 1].lower() if error_idx > 0 else "<s>"
    prev_2_word = sentence_words[error_idx - 2].lower() if error_idx > 1 else "<s>"
    next_word = sentence_words[error_idx + 1].lower() if error_idx < n_words - 1 else "</s>"
    next_2_word = sentence_words[error_idx + 2].lower() if error_idx < n_words - 2 else "</s>"

    # Chọn ngữ cảnh và tính khoảng cách cho đặc trưng similarity
    valid_context_words = []
    for i in range(local_start, local_end):
        if i == error_idx:
            continue
        if (i < error_idx or i not in error_indices) and sentence_words[i] not in stopwords:
            word = sentence_words[i]
            if word in word_to_idx:
                dist_weight = 1.0 / abs(i - error_idx) 
                valid_context_words.append((word, dist_weight))

    ctx_indices = [word_to_idx[ctx_word] for ctx_word, _ in valid_context_words]
    ctx_weights = [weight for _, weight in valid_context_words]

    # Tạo các ứng viên
    candidates = lookup(error_word, sym_dict, telex, word_to_idx, counts_1)
    
    # Gọi hàm đặc trưng similarity
    cand_to_sim = extract_similarity_features(
        candidates, ctx_indices, ctx_weights, word_to_idx, norm_embedding_matrix
    )

    top = []
    for candidate in candidates:
        candidate_lower = candidate.lower()
        
        # Tính đặc trưng 1: similarity
        weighted_sim = cand_to_sim.get(candidate, 0.0)
        norm_sim = max(0.0, weighted_sim) 

        # Gọi hàm đặc trưng 2: KenLM
        ken_score, norm_ken = extract_kenlm_feature(candidate, prefix_str, suffix_str, model_lm)

        # Gọi hàm đặc trưng 3: N-gram Counts
        c1, c2, c3, norm_c1, norm_c2, norm_c3 = extract_ngram_counts_feature(
            candidate_lower, prev_word, prev_2_word, next_word, next_2_word, counts_1, counts_2, counts_3
        )

        # Gọi hàm đặc trưng 4: Edit Distance (ĐÃ SỬA: Xóa bỏ dòng return lỗi chặn mạch tính toán)
        dist_val = edit_distance_telex(error_word, candidate, telex)
        norm_edit = 1.0 / (dist_val + 1)

        # Gọi hàm đặc trưng 5: Length Ratio
        length_ratio = extract_length_ratio_feature(error_word, candidate)

        # Tính total score -> ưu tiên các candidate có điểm tốt hơn
        total_score = (
            (0.30 * norm_ken) +    
            (0.25 * norm_edit) +  
            (0.10 * length_ratio) + 
            (0.20 * norm_c2) + 
            (0.05 * norm_c3) +     
            (0.05 * norm_c1) +
            (0.05 * norm_sim)      
        )
        
        top.append((total_score, candidate, ken_score, weighted_sim, c1, c2, c3, dist_val, length_ratio))

    # Sort để ưu tiên total score
    top.sort(key=lambda x: x[0], reverse=True)
    
    mock_candidates = []
    for item in top:
        (_, candidate, ken_score, weighted_sim, c1, c2, c3, dist_val, length_ratio) = item
        
        feature_vector = [ken_score, weighted_sim, c1, c2, c3, dist_val, length_ratio,]
        mock_candidates.append((candidate, feature_vector))

    return mock_candidates

In [25]:
sentence = "bữa trưa ăn bưởi trua"
sentence = sentence.split()
extract_candidates_and_features("trua", sentence, 4, [4])

[('chua',
  [-15.360492706298828, np.float64(0.0647602453827858), 43, 0, 0, 0.4, 1.0]),
 ('trau',
  [-17.32652473449707, np.float64(0.021328309550881386), 2, 0, 0, 0.5, 1.0]),
 ('trú',
  [-16.88465118408203, np.float64(0.048388510942459106), 53, 0, 0, 0.5, 0.75]),
 ('trưa',
  [-15.53065299987793, np.float64(0.3333333333333333), 38, 0, 0, 1.0, 1.0]),
 ('trâu',
  [-15.144950866699219, np.float64(0.1597086489200592), 69, 0, 0, 1.0, 1.0]),
 ('thua',
  [-15.005431175231934, np.float64(0.03368407487869263), 179, 0, 0, 1.0, 1.0]),
 ('truy',
  [-16.90864372253418, np.float64(0.027512136846780777), 170, 0, 0, 1.0, 1.0]),
 ('trí',
  [-16.631641387939453,
   np.float64(0.05801437050104141),
   267,
   0,
   0,
   1.0,
   0.75]),
 ('tra',
  [-16.782695770263672,
   np.float64(-0.004949700087308884),
   427,
   0,
   0,
   1.0,
   0.75]),
 ('trụ',
  [-16.214797973632812,
   np.float64(0.059984322637319565),
   131,
   0,
   0,
   1.0,
   0.75]),
 ('trúc',
  [-16.129934310913086, np.float64(0.017454

## 5.3 Lựa chọn ứng viên
Ta sẽ trích xuất đặc trưng của toàn bộ ứng viên trong tập df1_train['input'] để làm tài nguyên cho LightGBM học. \
Với mỗi từ sai thì ta sẽ sử dụng LightGBM để chọn ứng viên có điểm cao nhất làm từ sửa lỗi.\
Tạo hàm sửa lỗi hoàn chỉnh.
### 5.3.1 Tạo X_train, y_train, group_train cho LightGBM
Quét toàn bộ tập df1_train, với mỗi từ sai (input và target vị trí đó khác nhau), ta lưu cặp đó lại và vị trí của chúng. Là hàm find_misspelled_words_and_targets.\
Ta tạo các ứng viên cho từ lỗi, trích xuất đặc trưng của nó và lưu lại.\
Trong đó:
- X_train : lưu các đặc trưng của top n ứng viên dựa trên total_score (max = 21, 1 ứng viên đúng và nhiều nhất 20 ứng viên sai)
- y_train: lưu lại vị trí của ứng viên đúng là 1, còn lại là 0
- group_train: lưu lại có bao nhiêu ứng viên cho từ lỗi này (max = 21)\

Đã tạo dữ liệu cho LightGBM ở file create_data_LightGBM.ipynb

In [26]:
loaded_data = np.load('dataset_spell_correction.npz')
X_train = loaded_data['X_train']
y_train = loaded_data['y_train']
group_train = loaded_data['group_train']

### 5.3.2 Sử dụng LightGBM
Nạp data cho LightGBM, khởi tạo và huấn luyện nó

In [27]:
# Tắt toàn bộ các cảnh báo của sklearn/lightgbm
warnings.filterwarnings("ignore", category=UserWarning)

# 2. Khởi tạo Mô hình Ranker
ranker = lgb.LGBMRanker(
    objective='lambdarank',
    metric='ndcg',
    eval_at=[1, 3, 5],
    label_gain=[0, 1],
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=20, 
    random_state=42
)

# 3. Huấn luyện mô hình
ranker.fit(
    X=X_train,
    y=y_train,
    group=group_train
)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.028624 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1098
[LightGBM] [Info] Number of data points in the train set: 1066233, number of used features: 7


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.05
,n_estimators,100
,subsample_for_bin,200000
,objective,'lambdarank'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


Xem đóng góp của các feature với LightGBM

In [28]:
for feature_name, importance in zip(['ken_score', 'word2vec_sim', 'unigram_count', 'bigram_count', 'trigram_count', 'edit_dist', 'length_ratio', 'sym_rank'], ranker.feature_importances_):
    print(f"{feature_name}: {importance}")

ken_score: 292
word2vec_sim: 99
unigram_count: 588
bigram_count: 379
trigram_count: 214
edit_dist: 834
length_ratio: 594


### 5.3.3 Hàm sửa lỗi hoàn chỉnh
Với mỗi từ sai tìm được, nhờ LightGBM đưa ra ứng viên mà nó có điểm ranking là 1 để dự đoán

In [29]:
def correct_spelling_errors(sentence):
    error_indices = detect_error(sentence, model_lm)
    sentence_token = sentence.split()

    for idx in error_indices:
        error_word = sentence_token[idx]
        
        candidates_with_scores = extract_candidates_and_features(
            error_word, 
            sentence_token, 
            idx, 
            error_indices
        )

        if not candidates_with_scores:
            continue

        candidates = [item[0] for item in candidates_with_scores]
        candidates_score = [item[1] for item in candidates_with_scores]
    
        # Dự đoán với LightGBM
        scores = ranker.predict(candidates_score)
        
        # Lấy candidate có điểm cao nhất
        best_candidate = candidates[np.argmax(scores)]
        
        # Cập nhập sửa lỗi vào mảng
        # Lỗi tiếp theo sẽ đọc được từ 'best_candidate' này làm ngữ cảnh thay vì 'error_word'
        sentence_token[idx] = best_candidate
    
    return " ".join(sentence_token)

# 6. Đánh giá
## 6.1 Đánh giá giai đoạn bắt lỗi
Metric: $F_{0.5}\text{-Score}$\
Lý do: Thay vì dùng $F_1\text{-Score}$ (coi Precision và Recall quan trọng ngang nhau), $F_{0.5}$ sẽ đặt trọng số cho Precision cao gấp đôi Recall.\
Công thức:$$F_{0.5} = (1 + 0.5^2) \times \frac{\text{Precision} \times \text{Recall}}{(0.5^2 \times \text{Precision}) + \text{Recall}}$$\
Nếu điểm $F_{0.5}$ cao, chứng tỏ hệ thống hoạt động cực kỳ an toàn trên thực tế.

Hàm tìm các lỗi sai và vị trí lỗi sai trong câu:

In [31]:
def find_misspelled_words_and_targets(input_sentence, target_sentence):
    input_tokens = input_sentence.split()
    target_tokens = target_sentence.split()
    
    error_indices = []
    pairs = []
    if len(input_tokens) != len(target_tokens):
        return [], []

    for i in range(len(input_tokens)):
        if input_tokens[i] != target_tokens[i] and target_tokens[i] in word_to_idx:
            pairs.append((input_tokens[i], target_tokens[i]))
            error_indices.append(i)
            
    return pairs, error_indices

In [31]:
total_TP = 0  # True Positives: Máy dự đoán LỖI, và thực tế ĐÚNG LÀ LỖI
total_FP = 0  # False Positives: Máy dự đoán LỖI, nhưng thực tế KHÔNG PHẢI LỖI (Bắt nhầm)
total_FN = 0  # False Negatives: Thực tế LÀ LỖI, nhưng máy KHÔNG TÌM THẤY (Bỏ sót)

for idx, row in tqdm(df1_valid.iterrows(), total=len(df1_valid), desc="Đánh giá độ bắt lỗi trên tập Valid"):
    input_sent = str(row['input'])
    target_sent = str(row['target'])

    input_sent = replace_abbreviations(input_sent)

    # Tìm các từ lỗi và vị trí lỗi
    error_pairs, error_indices = find_misspelled_words_and_targets(input_sent, target_sent)
    
    # Bỏ qua câu không có lỗi
    if not error_pairs:
        continue 

    # Gọi hàm dự đoán của mô hình để sửa câu
    error_detect = detect_error(input_sent, model_lm)

    set_true = set(error_indices)
    set_pred = set(error_detect)

    # Tính TP, FP, FN cho câu hiện tại
    # 1. True Positives (Bắt trúng): Những index nằm trong cả 2 tập hợp
    TP = len(set_true & set_pred)
    
    # 2. False Positives (Bắt nhầm): Những index máy báo lỗi nhưng thực tế không có
    FP = len(set_pred - set_true)
    
    # 3. False Negatives (Bỏ sót): Những index thực tế có lỗi nhưng máy không báo
    FN = len(set_true - set_pred)

    # Cộng dồn vào tổng
    total_TP += TP
    total_FP += FP
    total_FN += FN

# Precision: Trong số các từ máy báo lỗi, có bao nhiêu % thực sự là lỗi?
precision = total_TP / (total_TP + total_FP) if (total_TP + total_FP) > 0 else 0

# Recall: Trong số toàn bộ các lỗi thực tế, máy tìm ra được bao nhiêu %?
recall = total_TP / (total_TP + total_FN) if (total_TP + total_FN) > 0 else 0

# F0.5-Score: Điểm trung bình điều hòa, chỉ số quan trọng nhất đánh giá Model
f05_score = (1 + 0.5**2) * (precision * recall) / ((0.5**2 * precision) + recall)

print(f"Số lỗi thực tế (Actual) : {total_TP + total_FN}")
print(f"Số lỗi dự đoán (Predict): {total_TP + total_FP}")
print(f" + Bắt trúng (True Positives) : {total_TP}")
print(f" + Bắt nhầm (False Positives): {total_FP}")
print(f" + Bỏ sót (False Negatives)  : {total_FN}\n")
print(f"Precision (Độ chính xác bắt lỗi): {precision * 100:.2f}%")
print(f"Recall (Độ phủ bắt lỗi)         : {recall * 100:.2f}%")
print(f"F0.5 (Điểm đánh giá cuối cùng)  : {f05_score * 100:.2f}%")

Đánh giá độ bắt lỗi trên tập Valid: 100%|██████████| 4458/4458 [00:00<00:00, 14632.13it/s]

Số lỗi thực tế (Actual) : 14612
Số lỗi dự đoán (Predict): 8883
 + Bắt trúng (True Positives) : 7963
 + Bắt nhầm (False Positives): 920
 + Bỏ sót (False Negatives)  : 6649

Precision (Độ chính xác bắt lỗi): 89.64%
Recall (Độ phủ bắt lỗi)         : 54.50%
F0.5 (Điểm đánh giá cuối cùng)  : 79.40%


## 6.2 Đánh giá giai đoạn xếp hạng ứng viên (Ranking)
Khâu này đánh giá xem mô hình LightGBM của bạn sắp xếp các ứng viên từ SymSpell tốt đến đâu.
- NDCG (Normalized Discounted Cumulative Gain): Bạn đã cấu hình sẵn trong LightGBM (metric='ndcg'). Chỉ số này đánh giá chất lượng của toàn bộ danh sách trả về. Nó thưởng điểm rất lớn nếu từ đúng được xếp ở Top 1, Top 2 và phạt nặng nếu xếp ở đáy.
- MRR (Mean Reciprocal Rank): Một metric cực kỳ chuẩn mực trong bài toán hệ thống gợi ý.
    - Nếu từ đúng nằm ở Top n $\rightarrow$ Điểm = 1/n.
    - Trung bình cộng điểm này của tất cả các lỗi sẽ ra MRR. Khác với NDCG phức tạp, MRR rất trực quan và dễ giải thích khi báo cáo.
- Top-K Hit Rate (Độ phủ Top-K): Tỉ lệ từ đúng nằm trong Top 1, Top 3, Top 5 của danh sách ứng viên.

In [52]:
# Khởi tạo các biến lưu trữ Ranking Metrics
count_error_all = 0
mrr_sum = 0.0
hit_at_1 = 0
hit_at_3 = 0
hit_at_5 = 0

for idx, row in tqdm(df1_valid.iterrows(), total=len(df1_valid), desc="Đánh giá Ranking (MRR, Hit@K)"):
    input_sent = str(row['input'])
    target_sent = str(row['target'])

    input_sent = replace_abbreviations(input_sent)

    input_tokens = input_sent.split()
    target_tokens = target_sent.split()

    # BẢO VỆ: Đảm bảo hai câu đồng bộ index
    if len(input_tokens) != len(target_tokens):
        continue

    # Dò lỗi bằng KenLM
    error_detect = detect_error(input_sent, model_lm)

    for actual_error_idx in error_detect:
        correct_word = target_tokens[actual_error_idx]
        error_word = input_tokens[actual_error_idx]
        
        # Bỏ qua nếu: máy bắt nhầm (từ gốc đã đúng), hoặc OOV ở target, hoặc là số
        if error_word == correct_word or correct_word not in vocab or correct_word.isdigit():
            continue

        # trích xuất tập ứng viên và đặc trưng của chúng
        candidates_with_features = extract_candidates_and_features(
            error_word, input_tokens, actual_error_idx, error_detect
        )

        if not candidates_with_features:
            count_error_all += 1 
            continue

        count_error_all += 1

        #LightGBM
        cand_words = [item[0] for item in candidates_with_features]
        X_infer = np.array([item[1] for item in candidates_with_features])
        
        # Lấy điểm dự đoán từ mô hình (giả sử tên biến model là lgbm_ranker)
        scores = ranker.predict(X_infer)

        # Sắp xếp danh sách từ theo điểm số giảm dần
        ranked_candidates = [word for _, word in sorted(zip(scores, cand_words), reverse=True)]

        # Tính MRR và HIT@K
        try:
            # Tìm vị trí của từ đúng trong danh sách (list.index trả về từ 0, nên cần +1)
            rank = ranked_candidates.index(correct_word) + 1
            
            # Cập nhật MRR: 1 chia cho thứ hạng
            mrr_sum += 1.0 / rank
            
            # Cập nhật Hit@K
            if rank == 1:
                hit_at_1 += 1
            if rank <= 3:
                hit_at_3 += 1
            if rank <= 5:
                hit_at_5 += 1
                
        except ValueError:
            # Nghĩa là correct_word KHÔNG NẰM TRONG danh sách ứng viên (Recall thất bại)
            # Không cộng điểm nào cho case này
            pass 

print("BÁO CÁO RANKING METRICS TRÊN TẬP VALID")
print("")
print(f"Tổng số lỗi đánh giá     : {count_error_all}")
print(f"MRR (Mean Reciprocal Rank) : {mrr_sum / count_error_all:.4f}")
print("")
print(f"Hit@1 (Top-1 Accuracy)     : {hit_at_1 / count_error_all:.4f}")
print(f"Hit@3 (Có trong Top 3)     : {hit_at_3 / count_error_all:.4f}")
print(f"Hit@5 (Có trong Top 5)     : {hit_at_5 / count_error_all:.4f}")


Đánh giá Ranking (MRR, Hit@K): 100%|██████████| 4458/4458 [55:09<00:00,  1.35it/s]     

BÁO CÁO RANKING METRICS TRÊN TẬP VALID

Tổng số lỗi đánh giá     : 7963
MRR (Mean Reciprocal Rank) : 0.6563

Hit@1 (Top-1 Accuracy)     : 0.5841
Hit@3 (Có trong Top 3)     : 0.6950
Hit@5 (Có trong Top 5)     : 0.7344


## 6.3 Đánh giá Word Accuracy
Khâu này đánh giá xem tỉ lệ các từ sau khi được sửa so với các từ đúng.

In [ ]:
count_error = 0
count_correct = 0

for idx, row in tqdm(df1_valid.iterrows(), total=len(df1_valid), desc="Đánh giá độ chính xác của các từ sai được sửa trên tập Valid"):
    input_sent = str(row['input'])
    target_sent = str(row['target'])

    input_sent = replace_abbreviations(input_sent)

    # Đưa CẢ CÂU qua toàn bộ pipeline thực tế (Detect -> Candidate -> Ranker -> Replace)
    # Tìm vị trí lỗi
    error_pairs, error_indices = find_misspelled_words_and_targets(input_sent, target_sent)
    
    if not error_pairs:
        continue # Bỏ qua câu không có lỗi

    # Gọi hàm dự đoán của mô hình để sửa câu
    # Giả định hàm này trả về 1 chuỗi string: "câu sau khi đã sửa"
    fixed_sentence_str = correct_spelling_errors(input_sent)
    
    fixed_tokens = fixed_sentence_str.split()
    input_tokens = input_sent.split()
    target_tokens = target_sent.split()

    if not (len(input_tokens) == len(target_tokens) == len(fixed_tokens)):
        continue

    error_detect = detect_error(input_sent, model_lm)
    for actual_error_idx in error_detect:
        if input_tokens[actual_error_idx] == target_tokens[actual_error_idx] or target_tokens[actual_error_idx] not in vocab:
            continue

        count_error += 1

        # Lấy từ đã được sửa ở đúng vị trí lỗi
        correct_word = target_tokens[actual_error_idx]
        fixed_word = fixed_tokens[actual_error_idx]

        # Kiểm tra xem máy sửa có khớp với đáp án target không
        if fixed_word == correct_word:
            count_correct += 1
            
accuracy = count_correct / count_error_all
    
print("Word Accuracy: ", count_correct / count_error_all)

Đánh giá độ chính xác của các từ sai được sửa trên tập Valid: 100%|██████████| 4458/4458 [27:47<00:00,  2.67it/s]  

Word Accuracy:  0.5997739545397464


## 6.4 Đánh giá Toàn cục (End-to-End Evaluation)
Đây là những con số quan trọng nhất để chứng minh giá trị thực tiễn của toàn bộ hệ thống.
- WER (Word Error Rate): So sánh trực tiếp câu đầu ra (sau khi đã sửa) với câu Target hoàn chỉnh, đếm tổng số phép Thêm (Insertions), Xóa (Deletions) và Thay thế (Substitutions) cần thiết để biến câu máy sửa thành câu chuẩn. Do hướng đi chỉ có sửa lỗi chính tả chứ không sửa khoảng cách nên ở đây chỉ tính phép thay thế.
    - Công thức:$$WER = \frac{S}{N}$$
    - Trong đó $N$ là tổng số từ của câu đích. WER càng thấp càng tốt (tiến về 0).

In [ ]:
total_word_errors = 0
total_reference_words = 0
exact_match_sentences = 0 # Đếm số câu hoàn hảo

for idx, row in tqdm(df1_valid.iterrows(), total=len(df1_valid), desc="Đánh giá End-to-End (WER)"):
    input_sent = str(row['input'])
    target_sent = str(row['target'])

    input_sent = replace_abbreviations(input_sent)

    # Đưa câu qua toàn bộ pipeline thực tế (Detect -> Candidate -> Ranker -> Replace)
    fixed_sentence_str = correct_spelling_errors(input_sent)
    
    # Tokenize (Tách từ)
    target_tokens = target_sent.split()
    fixed_tokens = fixed_sentence_str.split()
    
    # Đếm tổng số từ trong câu chuẩn (N)
    total_reference_words += len(target_tokens)
    
    # Tính số từ bị sai (S)
    errors_in_sentence = 0
    for idx in range(len(fixed_tokens)):
        if fixed_tokens[idx] != target_tokens[idx]:
            errors_in_sentence += 1
    total_word_errors += errors_in_sentence 
    
    # Nếu không có lỗi nào -> Câu được sửa hoàn hảo 100%
    if errors_in_sentence == 0:
        exact_match_sentences += 1

# In báo cáo kết quả
total_sentences = len(df1_valid)
overall_wer = (total_word_errors / total_reference_words) * 100

print("BÁO CÁO TOÀN CỤC END-TO-END (WER)")
print()
print(f"Tổng số câu test             : {total_sentences}")
print(f"Tổng số từ trong tập đích (N): {total_reference_words}")
print(f"Tổng số từ lỗi mô hình sinh ra: {total_word_errors}")
print()
# WER CÀNG THẤP CÀNG TỐT
print(f"WER (Word Error Rate)        : {overall_wer:.2f}%") 
# SER đo lường tỷ lệ câu được sửa hoàn hảo 100% 
print(f"SER (Sentence Exact Match)   : {(exact_match_sentences / total_sentences) * 100:.2f}%") 

Đánh giá End-to-End (WER): 100%|██████████| 4458/4458 [38:15<00:00,  1.94it/s]   

BÁO CÁO TOÀN CỤC END-TO-END (WER)

Tổng số câu test             : 4458
Tổng số từ trong tập đích (N): 58713
Tổng số từ lỗi mô hình sinh ra: 10610

WER (Word Error Rate)        : 18.07%
SER (Sentence Exact Match)   : 43.07%


# 7 trực quan:
Đưa ra 1 số câu sửa lỗi đúng và 1 số câu còn lỗi

In [52]:
exact_sentences = pd.DataFrame(columns=['Input', 'Fixed', 'Target'])
error_sentence = pd.DataFrame(columns=['Input', 'Fixed', 'Target'])
error_words = pd.DataFrame(columns=['Error', 'Correct'])

for idx, row in tqdm(df1_valid.iterrows(), total=len(df1_valid), desc="Tạo tập câu sửa lỗi đúng và còn sai"):
    input_sent = str(row['input'])
    target_sent = str(row['target'])

    input_sent = replace_abbreviations(input_sent)

    error_pairs, error_indices = find_misspelled_words_and_targets(input_sent, target_sent)

    # Đưa câu qua toàn bộ pipeline thực tế (Detect -> Candidate -> Ranker -> Replace)
    fixed_sentence_str = correct_spelling_errors(input_sent)
    
    # Tokenize (Tách từ)
    target_tokens = target_sent.split()
    fixed_tokens = fixed_sentence_str.split()
    
    errors_in_sentence = 0
    for idx in error_indices:
        if fixed_tokens[idx] != target_tokens[idx]:
            errors_in_sentence += 1
            i = error_words.shape[0]
            error_words.loc[i] = [fixed_tokens[idx], target_tokens[idx]]
    
    # Nếu không có lỗi nào -> Câu được sửa hoàn hảo 100%
    if errors_in_sentence == 0 and not error_pairs:
        i = exact_sentences.shape[0]
        exact_sentences.loc[i] = [str(row['input']), fixed_sentence_str, str(row['target'])]

    # Còn lỗi
    if errors_in_sentence != 0:
        i = error_sentence.shape[0]
        error_sentence.loc[i] = [str(row['input']), fixed_sentence_str, str(row['target'])]

Tạo tập câu sửa lỗi đúng và còn sai: 100%|██████████| 4458/4458 [23:23<00:00,  3.18it/s]  


## 7.1 Một số câu sửa lỗi hoàn chỉnh
Đây là những câu được sửa lỗi hoàn chỉnh, không sai từ nào

In [47]:
exact_sentences.head()

,Input,Fixed,Target
0,gửi con bạn mình ở đây chờ phần đọc típ,gửi con bạn mình ở đây chờ phần đọc tiếp,gửi con bạn mình ở đây chờ phần đọc tiếp
1,không để sai sót khi giới thiệu ứng viên đại b...,không để sai sót khi giới thiệu ứng viên đại b...,không để sai sót khi giới thiệu ứng viên đại b...
2,nguyễn nhung nhung ko để ý bộ tóc lắm,nguyễn nhung nhung không để ý bộ tóc lắm,nguyễn nhung nhung không để ý bộ tóc lắm
3,thầy mới mua cái áo vui quá,thầy mới mua cái áo vui quá,thầy mới mua cái áo vui quá
4,chúc mừng danh hài thầy giáo ba được đi all st...,chúc mừng danh hài thầy giáo ba được đi all st...,chúc mừng danh hài thầy giáo ba được đi all st...


## 7.2 Một số từ lỗi

In [48]:
error_words.head()

,Error,Correct
0,một,đối
1,tủ,thủ
2,nà,là
3,tro,trợ
4,truc,trực


## 7.3 Một số câu vẫn còn lỗi sau khi sửa
Đây là những câu mà sau khi sửa vẫn còn lỗi

In [53]:
error_sentence.head()

,Input,Fixed,Target
0,ốj tủ owr tuws eest của djokovic nà jannik sinner,một tủ ở tứ kết của djokovic nà jannik sinner,đối thủ ở tứ kết của djokovic là jannik sinner
1,facebook hỗ tro office truc tuyen,facebook chỗ tro office truc tuyen,facebook hỗ trợ office trực tuyến
2,vì sao honsg nên đi du lịck tkái lacn mùa lễ,vì sao ông nên đi du lịch tại sân mùa lễ,vì sao không nên đi du lịch thái lan mùa lễ
3,vì ló khôgn côngm bằgng,vì ló khôgn công bằng,vì nó không công bằng
4,sống vật vờ đến tuổi r chet,sống vật vờ đến tuổi rồi nhé,sống vật vờ đến tuổi rồi chết
